# 5) Initial Data Preprocessing & Dataset Documentation

This notebook performs the initial preprocessing and documentation of the healthcare disease prediction dataset.

The main objectives are:

- Load the dataset safely
- Inspect the original dataset structure
- Standardize column names
- Handle empty and missing symptom values
- Remove exact duplicate records
- Standardize text values
- Analyze symptom naming consistency
- Document the dataset structure
- Save the initially processed dataset

This stage prepares the dataset for further feature engineering and machine learning preprocessing.

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
import requests
import csv
from io import StringIO

## 1. Load the Original Dataset

In [ ]:
# GitHub raw dataset URL

url = "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv"

# Download dataset
response = requests.get(url)
response.raise_for_status()

# Read CSV using Python csv module
reader = csv.reader(StringIO(response.text))
rows = list(reader)

# Separate header and data
header = rows[0]
data_rows = rows[1:]

# Find maximum number of fields
max_fields = max(len(row) for row in data_rows)

# Add missing column names if required
while len(header) < max_fields:
    header.append(f"Symptom_{len(header)}")

# Make every row the same length
fixed_rows = []

for row in data_rows:
    if len(row) < max_fields:
        row = row + [""] * (max_fields - len(row))
    elif len(row) > max_fields:
        row = row[:max_fields]
    fixed_rows.append(row)

# Create DataFrame
df = pd.DataFrame(fixed_rows, columns=header)

print("Dataset loaded successfully!")
print("Original shape:", df.shape)

In [ ]:
# Display original dataset
df.head()

## 2. Identify Dataset Columns

In [ ]:
disease_column = "Disease"

symptom_columns = [
    column for column in df.columns
    if column.startswith("Symptom_")
]

print("Target column:", disease_column)
print("Number of symptom columns:", len(symptom_columns))

print("\nColumns:")
for column in df.columns:
    print("-", column)

## 3. Initial Dataset Documentation

In [ ]:
# Create dataset documentation table

documentation = pd.DataFrame({
    "Column": df.columns,
    "Data Type": [df[column].dtype for column in df.columns],
    "Non-Null Values": [df[column].notna().sum() for column in df.columns],
    "Empty Values": [(df[column] == "").sum() for column in df.columns],
    "Unique Values": [df[column].nunique(dropna=True) for column in df.columns]
})

documentation

## 4. Standardize Column Names

In [ ]:
# Remove unnecessary spaces from column names
df.columns = df.columns.str.strip()

print("Column names after standardization:")
print(df.columns.tolist())

## 5. Handle Empty Values

Empty strings in symptom columns are converted to NaN so that they can be handled consistently during later preprocessing.

In [ ]:
# Convert blank and whitespace-only values to NaN

df = df.replace(r"^\s*$", np.nan, regex=True)

print("Empty values have been converted to NaN.")

missing_counts = df.isnull().sum()
missing_counts

In [ ]:
# Calculate missing-value summary

missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

missing_summary

## 6. Standardize Text Values

In [ ]:
# Strip unnecessary spaces from text columns

text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = df[column].apply(
        lambda value: value.strip() if isinstance(value, str) else value
    )

print("Whitespace has been removed from text values.")

## 7. Standardize Disease Names

In [ ]:
# Standardize disease text formatting

df[disease_column] = df[disease_column].apply(
    lambda value: value.title() if isinstance(value, str) else value
)

print("Disease names have been standardized.")
print("Unique diseases:", df[disease_column].nunique())

## 8. Symptom Naming Analysis

Symptoms may contain different names or similar terms. These values are documented before applying any final symptom standardization.

In [ ]:
# Collect all available symptoms

all_symptoms = pd.concat(
        [df[column] for column in symptom_columns],
        ignore_index=True
        )

all_symptoms = all_symptoms.dropna()

symptom_counts = all_symptoms.value_counts()

print("Total symptom occurrences:", len(all_symptoms))
print("Unique symptom values:", all_symptoms.nunique())

In [ ]:
# Display symptom frequencies
        
        symptom_frequency = symptom_counts.reset_index()
        symptom_frequency.columns = ["Symptom", "Frequency"]
        
        symptom_frequency.head(30)

## 9. Standardize Symptom Text Formatting

In [ ]:
# Standardize capitalization and whitespace in symptoms

for column in symptom_columns:
    df[column] = df[column].apply(
        lambda value: value.strip().title()
        if isinstance(value, str)
        else value
    )

print("Symptom text formatting has been standardized.")

## 10. Check Duplicate Records

In [ ]:
# Count duplicate records

duplicate_count = df.duplicated().sum()

print("Duplicate records before removal:", duplicate_count)

In [ ]:
# Remove exact duplicate records

before_duplicates = len(df)
        
        df = df.drop_duplicates().reset_index(drop=True)
        
        after_duplicates = len(df)
        removed_duplicates = before_duplicates - after_duplicates
        
        print("Records before duplicate removal:", before_duplicates)
        print("Records after duplicate removal :", after_duplicates)
        print("Duplicates removed              :", removed_duplicates)

## 11. Analyze Incomplete Records

In [ ]:
# Count available symptoms per record

        symptom_count = df[symptom_columns].notna().sum(axis=1)

print("Minimum symptoms per record:", symptom_count.min())
print("Maximum symptoms per record:", symptom_count.max())
print("Average symptoms per record:", round(symptom_count.mean(), 2))

In [ ]:
# Identify incomplete records
        
        incomplete_records = df[symptom_count < len(symptom_columns)]
        complete_records = df[symptom_count == len(symptom_columns)]
        
        print("Complete records:", len(complete_records))
        print("Incomplete records:", len(incomplete_records))
        print(
        "Incomplete percentage:",
        round(len(incomplete_records) / len(df) * 100, 2),
        "%"
        )

## 12. Verify Target Variable

In [ ]:
# Check target variable
        
        print("Target variable:", disease_column)
        print("Number of disease classes:", df[disease_column].nunique())
        print("Missing target values:", df[disease_column].isnull().sum())
        
        print("\nTop disease classes:")
        print(df[disease_column].value_counts().head(20))

## 13. Final Preprocessing Validation

In [ ]:
# Validate processed dataset
        
        print("FINAL DATASET VALIDATION")
        print("=" * 50)
        print("Rows:", df.shape[0])
        print("Columns:", df.shape[1])
        print("Missing values:", df.isnull().sum().sum())
        print("Duplicate rows:", df.duplicated().sum())
        print("Disease classes:", df[disease_column].nunique())
        print("Symptom columns:", len(symptom_columns))

In [ ]:
# Final column documentation
        
        final_documentation = pd.DataFrame({
        "Column": df.columns,
        "Role": [
        "Target" if column == disease_column else "Feature"
        for column in df.columns
        ],
        "Data Type": [df[column].dtype for column in df.columns],
        "Unique Values": [
        df[column].nunique(dropna=True)
        for column in df.columns
        ],
        "Missing Values": [
        df[column].isnull().sum()
        for column in df.columns
        ]
        })
        
        final_documentation

## 14. Save Initially Processed Dataset

In [ ]:
# Save the initially processed dataset
        
        output_file = "initial_preprocessed_dataset.csv"
        
        df.to_csv(output_file, index=False)
        
        print("Processed dataset saved successfully!")
        print("File:", output_file)

## Dataset Documentation Summary

### Dataset Purpose
The dataset is used for disease prediction based on reported symptoms.

### Target Variable
**Disease** is the target variable that represents the disease class to be predicted.

### Feature Variables
**Symptom_1** through the available symptom columns represent symptoms associated with each disease record.

### Initial Preprocessing Performed
- Empty values converted to missing values
- Column names standardized
- Whitespace removed from text values
- Disease names standardized for capitalization
- Symptom text formatting standardized
- Exact duplicate records removed
- Incomplete symptom records identified and retained
- Dataset structure documented

### Important Note
Incomplete records are not automatically deleted because a record with fewer symptoms can still contain useful information for disease prediction. Further feature engineering and encoding will be performed in later preprocessing stages.

# Conclusion

Initial data preprocessing and dataset documentation were completed successfully. The dataset was loaded safely, text values were standardized, empty values were converted to missing values, duplicate records were removed, and incomplete symptom records were identified.

The processed dataset is now ready for the next stages of feature engineering, symptom representation, encoding, and machine learning model preparation.